In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

C:\Users\PC 2026\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
model_id = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16,
    device_map="auto"
)

`torch_dtype` is deprecated! Use `dtype` instead!
C:\Users\PC 2026\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):
W0102 17:24:16.945000 15192 torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
C:\Users\PC 2026\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\PC 2026\.cache\huggingface\hub\models--mistralai--Mistral-7B-Instruct-v0.2. Caching files will still work but in a degraded version that might require more space

In [4]:
prompt = "Explain what machine learning is in simple terms."

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

In [5]:
outputs = model.generate(
    **inputs,
    max_new_tokens=150,
    temperature=0.7,
    do_sample=True
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Explain what machine learning is in simple terms. Machine learning is a type of artificial intelligence (AI) that allows computers to learn and improve from experience without being explicitly programmed. It's like teaching a child to recognize animals by showing them pictures and labeling them, and then having the child identify new animals on their own based on what they've learned. In machine learning, computers are given large amounts of data and use algorithms to learn patterns and make predictions or decisions without being explicitly programmed to perform the task.


In [6]:
with torch.no_grad():
    outputs = model(**inputs)

In [8]:
# Logits for the last token
logits = outputs.logits[:, -1, :]  # shape: [1, vocab_size]
logits, logits.shape

(tensor([[-7.9648, -8.5391, -1.7461,  ..., -4.5859, -4.7852, -3.2617]],
        dtype=torch.float16),
 torch.Size([1, 32000]))

In [24]:
# Convert to probabilities
teacher_distribution = torch.softmax(logits, dim=-1)
VOC_SIZE = teacher_distribution.shape[-1]
teacher_distribution, teacher_distribution.shape

(tensor([[0., 0., 0.,  ..., 0., 0., 0.]], dtype=torch.float16),
 torch.Size([1, 32000]))

In [22]:
# Top-k probabilities
top_k = 10
top_probs, top_ids = torch.topk(teacher_distribution, top_k)

for prob, token_id in zip(top_probs[0], top_ids[0]):
    token = tokenizer.decode(token_id)
    print(f"{token!r}: {prob.item():.4f}")

'\n': 0.5127
'Machine': 0.4456
'': 0.0175
'How': 0.0040
'In': 0.0027
'I': 0.0023
'What': 0.0014
'Machine': 0.0012
'Sure': 0.0012
'Can': 0.0010


In [16]:
import numpy as np

EPS = 1e-12  # numerical stability

In [ ]:
def generate_sparse_logits(
    vocab_size=VOC_SIZE,
    non_zero_count=50,
    scale=1.0,
    seed=None
):
    """
    Generate sparse LLM-style logits with shape [1, vocab_size].
    Most values are 0.
    """
    if seed is not None:
        np.random.seed(seed)

    logits = np.zeros((1, vocab_size), dtype=np.float32)

    indices = np.random.choice(vocab_size, non_zero_count, replace=False)
    logits[0, indices] = np.random.normal(0.0, scale, size=non_zero_count)

    return logits

In [26]:
student_distribution = generate_sparse_logits()
student_distribution, student_distribution.shape

(array([[0., 0., 0., ..., 0., 0., 0.]], shape=(1, 32000), dtype=float32),
 (1, 32000))

In [28]:

def _normalize(p):
    p = np.asarray(p, dtype=float)
    p = np.clip(p, EPS, None)
    return p / p.sum()

# 0. Kullback–Leibler Divergence (KL(p || q))
def kullback_leibler_divergence(p, q):
    p = _normalize(p)
    q = _normalize(q)
    return np.sum(p * np.log(p / q))

def symmetric_kl_divergence(p, q):
    return (
        kullback_leibler_divergence(p, q) +
        kullback_leibler_divergence(q, p)
    ) / 2

# 1. Jensen–Shannon Divergence
def jensen_shannon_divergence(p, q):
    p = _normalize(p)
    q = _normalize(q)
    m = 0.5 * (p + q)
    kl_pm = np.sum(p * np.log(p / m))
    kl_qm = np.sum(q * np.log(q / m))
    return 0.5 * (kl_pm + kl_qm)


# 2. Wasserstein Distance (1D, discrete)
def wasserstein_distance(p, q):
    p = _normalize(p)
    q = _normalize(q)
    cdf_p = np.cumsum(p)
    cdf_q = np.cumsum(q)
    return np.sum(np.abs(cdf_p - cdf_q))


# 3. Rényi Divergence (order alpha)
def renyi_divergence(p, q, alpha=2.0):
    if alpha <= 0 or alpha == 1:
        raise ValueError("alpha must be > 0 and != 1")
    p = _normalize(p)
    q = _normalize(q)
    return (1 / (alpha - 1)) * np.log(np.sum(p**alpha * q**(1 - alpha)))


# 4. Hellinger Distance
def hellinger_distance(p, q):
    p = _normalize(p)
    q = _normalize(q)
    return (1 / np.sqrt(2)) * np.linalg.norm(np.sqrt(p) - np.sqrt(q))


# 5. Bhattacharyya Distance
def bhattacharyya_distance(p, q):
    p = _normalize(p)
    q = _normalize(q)
    bc = np.sum(np.sqrt(p * q))
    return -np.log(bc)


# Example usage
if __name__ == "__main__":
    p = teacher_distribution[0]
    q = student_distribution[0]

    print("KL(p || q):", kullback_leibler_divergence(p, q))
    print("KL(q || p):", kullback_leibler_divergence(q, p))
    print("JSD:", jensen_shannon_divergence(p, q))
    print("Wasserstein:", wasserstein_distance(p, q))
    print("Rényi (α=2):", renyi_divergence(p, q, alpha=2))
    print("Hellinger:", hellinger_distance(p, q))
    print("Bhattacharyya:", bhattacharyya_distance(p, q))


KL(p || q): 29.54713695800313
KL(q || p): 20.23816829298822
JSD: 0.6930620412091196
Wasserstein: 14626.408249279273
Rényi (α=2): 29.729196640378976
Hellinger: 0.9991705618488174
Bhattacharyya: 6.402029637213286
